In [28]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import easyocr

reader = easyocr.Reader(['en'])

def show(img, title):
    """Helper to display each stage"""
    plt.figure(figsize=(4,3))
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()
    
def preprocess(image_path, debug=True):
    import cv2

    img = cv2.imread(image_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    resized = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    return resized

# def preprocess(image_path, debug=True):
    
    # img = cv2.imread(image_path)

    # if img is None:
    #     print("Image not found")
    #     return None

    # if debug:
    #     show(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), "Original")

    # # -----------------------------
    # # STEP 1: GRAYSCALE
    # # -----------------------------
    # gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # print("✔ Step 1: Grayscale done")

    # if debug:
    #     show(gray, "Grayscale")

    # # -----------------------------
    # # STEP 2: UPSCALE (helps small text)
    # # -----------------------------
    # resized = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    # print("✔ Step 2: Resized")

    # if debug:
    #     show(resized, "Resized")

    # # -----------------------------
    # # STEP 3: LIGHT NOISE REMOVAL
    # # (kept mild so we don't destroy text)
    # # -----------------------------
    # denoised = cv2.bilateralFilter(resized, 7, 50, 50)
    # print("✔ Step 3: Denoised")

    # if debug:
    #     show(denoised, "Denoised")

    # # -----------------------------
    # # STEP 4: CONTRAST BOOST
    # # -----------------------------
    # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    # contrasted = clahe.apply(denoised)
    # print("✔ Step 4: Contrast enhanced")

    # if debug:
    #     show(contrasted, "CLAHE Contrast")

    # # -----------------------------
    # # STEP 5: FINAL OUTPUT (NO THRESHOLD)
    # # IMPORTANT: EasyOCR prefers grayscale
    # # -----------------------------
    # print("✔ Step 5: Ready for OCR")

    # if debug:
    #     show(contrasted, "FINAL OCR INPUT")

    # return contrasted


# def run_ocr(image_path):
#     processed = preprocess(image_path, debug=True)

#     if processed is None:
#         return

#     print("\n🤖 Running EasyOCR...\n")

#     # EasyOCR can take numpy array directly (NO need to save temp file)
#     result = reader.readtext(processed, detail=0)

#     print("🔍 OCR OUTPUT:")
#     print(result)

#     return result

def run_ocr(image_path):
    import cv2

    img = cv2.imread(image_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # upscale
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # try BOTH original + enhanced (important trick)
    result = reader.readtext(gray, detail=0)

    print("🔍 OCR OUTPUT:", result)

    text = clean_text(result)
    all_dates = extract_all_dates(text)

    print("🧼 CLEAN TEXT:", text)
    print("📅 DATES FOUND:", all_dates)

    return text, all_dates

In [30]:
import re
from datetime import datetime

# -----------------------------
# CLEAN OCR OUTPUT (ROBUST)
# -----------------------------
def clean_text(result_list):
    text = " ".join(result_list)

    # Fix broken digit splits like "0 9 / 2 5" → "09/25"
    text = re.sub(r'(\d)\s+(\d)', r'\1\2', text)

    # Fix broken alphabets like "OC T" → "OCT"
    text = re.sub(r'\b([A-Z]{1,2})\s([A-Z]{1,2})\b', r'\1\2', text)

    return text.upper()


# -----------------------------
# EXTRACT ALL POSSIBLE DATES
# -----------------------------
def extract_all_dates(text):
    patterns = [
        r'\b\d{2}[./-]\d{2}[./-]\d{4}\b',
        r'\b\d{2}[./-]\d{2}[./-]\d{2}\b',
        r'\b\d{2}[./-]\d{4}\b',
        r'\b(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*\.?\s*\d{2,4}\b',
        r'\b\d{6}\b',
        r'\b\d{8}\b',
    ]

    found = []
    for p in patterns:
        found.extend(re.findall(p, text))

    return found


# -----------------------------
# NORMALIZE STRING FOR MATCHING
# (VERY IMPORTANT FIX)
# -----------------------------
def normalize(s):
    return re.sub(r'[^A-Z0-9]', '', s)


# -----------------------------
# FIND EXPIRY NEAR KEYWORDS
# -----------------------------
def find_expiry_in_context(text, all_dates):
    exp_pattern = r'(EXP(?:IRY)?|BEST\s*BEFORE|USE\s*BY|BB)[^\w]*([^\n]{1,40})'
    match = re.search(exp_pattern, text)

    if match:
        nearby_text = match.group(2)
        nearby_clean = normalize(nearby_text)

        for d in all_dates:
            if normalize(d) in nearby_clean:
                return d

    return None


# -----------------------------
# PARSE DATE STRING → DATETIME
# -----------------------------
def parse_date(date_str):
    formats = [
        "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y",
        "%d/%m/%y", "%d-%m-%y", "%d.%m.%y",
        "%m/%Y", "%m-%Y", "%m.%Y",
        "%b %Y", "%B %Y",
        "%b%Y", "%b %y", "%b%y",
    ]

    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except:
            continue

    if re.fullmatch(r'\d{6}', date_str):
        try:
            return datetime.strptime(date_str, "%d%m%y")
        except:
            pass

    if re.fullmatch(r'\d{8}', date_str):
        try:
            return datetime.strptime(date_str, "%d%m%Y")
        except:
            pass

    return None


# -----------------------------
# MAIN OCR PIPELINE (FIXED)
# -----------------------------
def run_ocr(image_path):
    processed = preprocess(image_path, debug=True)

    if processed is None:
        print("❌ Preprocessing failed")
        return None, None

    print("\n🤖 Running EasyOCR...\n")

    # IMPORTANT: direct numpy input (no temp file)
    result = reader.readtext(processed, detail=0)

    print("🔍 RAW OCR OUTPUT:", result)

    text = clean_text(result)

    print("🧼 CLEAN TEXT:", text)

    all_dates = extract_all_dates(text)

    print("📅 DATES FOUND:", all_dates)

    return text, all_dates

In [15]:
# import easyocr
# import re
# from datetime import datetime

# reader = easyocr.Reader(['en'])

# # Clean OCR garbage
# def clean_text(result_list):
#     text = " ".join(result_list)
    
#     # Fix broken words like "OC T" → "OCT"
#     text = re.sub(r'\b([A-Z]{1,2})\s([A-Z]{1,2})\b', r'\1\2', text)
    
#     return text.upper()


# # Extract all possible date formats
# def extract_all_dates(text):
#     patterns = [
#         r'\b\d{2}[./-]\d{2}[./-]\d{4}\b',
#         r'\b\d{2}[./-]\d{2}[./-]\d{2}\b',
#         r'\b\d{2}[./-]\d{4}\b',
#         r'\b(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*\.?\s*\d{2,4}\b',
#         r'\b\d{6}\b',
#         r'\b\d{8}\b',
#     ]
    
#     found = []
#     for p in patterns:
#         found.extend(re.findall(p, text))
    
#     return found


# # Find expiry near keyword
# def find_expiry_in_context(text, all_dates):
#     exp_pattern = r'(?:EXP(?:IRY)?|BEST\s*BEFORE|USE\s*BY|BB)[^\w]*([^\n]{1,30})'
#     match = re.search(exp_pattern, text)
    
#     if match:
#         nearby_text = match.group(1)
#         for d in all_dates:
#             if d in nearby_text:
#                 return d
    
#     return None


# # Convert string → datetime
# def parse_date(date_str):
#     formats = [
#         "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y",
#         "%d/%m/%y", "%d-%m-%y", "%d.%m.%y",
#         "%m/%Y", "%m-%Y", "%m.%Y",
#         "%b %Y", "%B %Y",
#         "%b%Y", "%b %y", "%b%y",
#     ]
    
#     for fmt in formats:
#         try:
#             return datetime.strptime(date_str, fmt)
#         except:
#             continue

#     if re.fullmatch(r'\d{6}', date_str):
#         try:
#             return datetime.strptime(date_str, "%d%m%y")
#         except:
#             pass

#     if re.fullmatch(r'\d{8}', date_str):
#         try:
#             return datetime.strptime(date_str, "%d%m%Y")
#         except:
#             pass
    
#     return None


# # OCR PIPELINE
# def run_ocr(image_path):
#     processed = preprocess(image_path)
    
#     temp_path = "temp_processed.png"
#     cv2.imwrite(temp_path, processed)
    
#     result = reader.readtext(temp_path, detail=0)
#     text = clean_text(result)
    
#     print("🔍 OCR TEXT:", text)
    
#     all_dates = extract_all_dates(text)
#     print("📅 Dates found:", all_dates)
    
#     return text, all_dates

In [31]:
from datetime import datetime

def check_expiry(image_path):
    text, all_dates = run_ocr(image_path)
    
    # Step 1: try near EXP keyword
    expiry_str = find_expiry_in_context(text, all_dates)
    
    # Step 2: fallback → latest date
    if not expiry_str:
        parsed = [parse_date(d) for d in all_dates]
        parsed = [d for d in parsed if d is not None]
        
        if not parsed:
            return "❌ No valid date found"
        
        expiry = max(parsed)
        method = "latest date fallback"
    else:
        expiry = parse_date(expiry_str)
        
        if not expiry:
            return f"❌ Found '{expiry_str}' but couldn't parse it"
        
        method = "found near EXP"
    
    print("⚙️ Method:", method)
    
    if expiry < datetime.today():
        return f"❌ Expired on {expiry.strftime('%b %Y')}"
    else:
        return f"✅ Valid until {expiry.strftime('%b %Y')}"


# RUN
image_path = "/Users/prekshyajoshi/instacart_recommender_project/ohno.png"

result = check_expiry(image_path)

print("-" * 40)
print("FINAL RESULT:", result)


🤖 Running EasyOCR...

🔍 RAW OCR OUTPUT: ['ENcak ur', 'ee  enen', 'L', 'ltxuetee t', 'Meraul MrA dleodse', 'baati tEna', 'Hie ] etut', 'cuu', 'Jm', 'E0 # Mgleci Iigaie', 'Knauuer', '4ehaml', 'ne', 'Apeze017440049 0 concelte * / O', 'Tea', 'WateCa', '5444 # 4 nneoication Conala', 'Peaae pu', 'LAel', 'maaty', 'Iloljdt Adroit ', 'W', 'u9', 'FIie Karana Zaillian', 'M', 'Anllete', '187ta', 'Road Jlclon', 'ul', '(ndl}', 'Ki"', 'Sa172MP)', 'Esat Lc Nal', '0', 'K', '{3', 'ai:', 'ntoeasadbiore', 'MEFO25046', '6', '3iz  2328', '{88', '6', '83z8', 'pl', 'd', 'banedlid', 'under Regiatralion ol Sepor (', 'TAA', 'Moulua', 'ad', '(eet', 'Fanel', 'noe', 'uut', 't', 'rlrt', '(h', 'Biomedli', 'an', 'LBScST', 'Mercea', 'Trn1slooe', 'Brc', '26', 'Juca', '#par', '0007u:', 'Ded', 'Dac', 'qqlaui', '<', 'A1o91bur _uu-', 'euuLett [', 'duaea', 'Edbn', '4"', 'L Aera', 'Te', 'Atanent', 'Aa', 'Adoi [', '(Fiane)', 'Aedunalt']
🧼 CLEAN TEXT: ENCAK UR EE  ENEN L LTXUETEE T MERAUL MRA DLEODSE BAATI TENA HIE ] ETUT CUU 

In [23]:
def preprocess(image_path, debug=True):
    import cv2

    img = cv2.imread(image_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    resized = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    return resized